# CLIC 2027 nhánh B: DC-AE hỏng ở đâu trên tập validation

**Cài đặt:** Accelerator = **GPU T4 x1**, Internet = **On**. Mất khoảng 40–60 phút.

Các bước: kiểm kê và OCR; tái tạo bằng trần DC-AE, trần SD-VAE f8 và HEVC-intra ở 0.075/0.15/0.3 bpp; tính PSNR, MS-SSIM, LPIPS, DISTS, text-PSNR, OCR CER; cắt crop; bảng quyết định B3.

**Kết quả:** `b_results.zip`, đặt vào `results/clic_b/`.

*Sinh bởi `notebooks/build.py` từ commit `ff464ef` của repo; đừng sửa tay notebook này.*

In [ ]:
!mkdir -p experiments/clic_b src/ratflow src/ratflow/eval

In [ ]:
%%writefile src/ratflow/__init__.py
"""Rate-as-Time flow-bridge image codec (CVPR 2027) and its CLIC 2027 submission."""


In [ ]:
%%writefile src/ratflow/eval/__init__.py


In [ ]:
%%writefile src/ratflow/eval/metrics.py
"""Shared evaluation code: every number in the paper and in the CLIC analyses goes through here,
so ceilings, baselines and our codec stay comparable.

Conventions
- Images are HxWx3 uint8 numpy arrays.
- PSNR per image from the RGB MSE. Dataset-level "CLIC PSNR" pools MSE weighted by pixel count.
- LPIPS (alex, inputs in [-1, 1]) and DISTS are computed on 512x512 tiles, weighted by tile area
  (tiles with a side < 64 are skipped), so 2K+ images fit in GPU memory.
- MS-SSIM on the full image (NaN if the short side is < 161, the minimum for 5 scales).
- FID/KID on 256x256 patches, HiFiC protocol: a grid plus a grid shifted by 128 in both directions.
- OCR CER: easyocr on the same crops (boxes detected on the original) for reference and reconstruction.

torch, lpips, piq, torchmetrics and easyocr are imported lazily so the numpy helpers work without them.
"""
from __future__ import annotations

import csv
import math
from pathlib import Path

import numpy as np
from PIL import Image

TILE = 512
FID_PATCH = 256
IMG_EXTS = (".png", ".jpg", ".jpeg")


# ---------------------------------------------------------------- io
def load(p) -> np.ndarray:
    return np.asarray(Image.open(p).convert("RGB"))


def save(a: np.ndarray, p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(a).save(p)


def images(d: Path, exts=IMG_EXTS):
    """Sorted image files under d, skipping macOS zip debris (__MACOSX/, ._*)."""
    return sorted(p for p in Path(d).rglob("*") if p.suffix.lower() in exts
                  and not p.name.startswith("._") and "__MACOSX" not in p.parts)


def write_csv(rows, path):
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, keys)
        w.writeheader()
        w.writerows(rows)


def read_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))


# ---------------------------------------------------------------- torch helpers
def device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def to_tensor(a: np.ndarray, dev):
    """HxWx3 uint8 -> 1x3xHxW float in [0, 1]."""
    import torch
    return torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1)[None].float().div(255).to(dev)


# ---------------------------------------------------------------- fidelity
def mse(a: np.ndarray, b: np.ndarray) -> float:
    return float(((a.astype(np.float64) - b) ** 2).mean())


def psnr_from_mse(m: float) -> float:
    return 10 * math.log10(255 ** 2 / max(m, 1e-10))


def psnr(a, b) -> float:
    return psnr_from_mse(mse(a, b))


def pooled_psnr(mses, pixels) -> float:
    """CLIC-style dataset PSNR: MSE averaged with pixel-count weights."""
    return psnr_from_mse(sum(m * n for m, n in zip(mses, pixels)) / sum(pixels))


def msssim(a, b, dev) -> float:
    import piq
    import torch
    if min(a.shape[:2]) < 161:
        return float("nan")
    with torch.no_grad():
        return float(piq.multi_scale_ssim(to_tensor(a, dev), to_tensor(b, dev), data_range=1.0))


# ---------------------------------------------------------------- perceptual
def tiled(fn, a, b, dev, tile=TILE) -> float:
    """Area-weighted mean of fn(tile_a, tile_b) over tile x tile crops; fn takes [0,1] tensors."""
    import torch
    H, W = a.shape[:2]
    tot = wsum = 0.0
    for y in range(0, H, tile):
        for x in range(0, W, tile):
            pa, pb = a[y:y + tile, x:x + tile], b[y:y + tile, x:x + tile]
            if min(pa.shape[:2]) < 64:
                continue
            n = pa.shape[0] * pa.shape[1]
            with torch.no_grad():
                tot += float(fn(to_tensor(pa, dev), to_tensor(pb, dev))) * n
            wsum += n
    return tot / wsum


class Perceptual:
    """LPIPS-alex and DISTS with the tiling convention above."""

    def __init__(self, dev=None):
        import lpips
        import piq
        self.dev = dev or device()
        self._lpips = lpips.LPIPS(net="alex", verbose=False).to(self.dev)
        self._dists = piq.DISTS().to(self.dev)

    def lpips(self, a, b) -> float:
        return tiled(lambda x, y: self._lpips(x * 2 - 1, y * 2 - 1).mean(), a, b, self.dev)

    def dists(self, a, b) -> float:
        return tiled(lambda x, y: self._dists(x, y), a, b, self.dev)

    def all(self, a, b) -> dict:
        m = mse(a, b)
        return dict(mse=m, psnr=psnr_from_mse(m), msssim=msssim(a, b, self.dev),
                    lpips=self.lpips(a, b), dists=self.dists(a, b))


# ---------------------------------------------------------------- FID / KID
def patches(a: np.ndarray, size=FID_PATCH):
    """HiFiC protocol: size x size grid, plus the grid shifted by size // 2."""
    H, W = a.shape[:2]
    for off in (0, size // 2):
        for y in range(off, H - size + 1, size):
            for x in range(off, W - size + 1, size):
                yield a[y:y + size, x:x + size]


def fid_kid(pairs, dev=None, batch=64, kid_subset=1000) -> dict:
    """pairs: iterable of (original, reconstruction) uint8 arrays. Returns fid, kid, kid_std, n_patches."""
    import torch
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.kid import KernelInceptionDistance
    dev = dev or device()
    f = FrechetInceptionDistance(feature=2048, normalize=False).to(dev)
    k = KernelInceptionDistance(subset_size=kid_subset, normalize=False).to(dev)

    def push(ps, real):
        t = torch.from_numpy(np.stack(ps)).permute(0, 3, 1, 2).to(dev)  # uint8 NCHW
        f.update(t, real=real)
        k.update(t, real=real)

    n = 0
    for x, y in pairs:
        pr, pf = list(patches(x)), list(patches(y))
        for i in range(0, len(pr), batch):
            push(pr[i:i + batch], True)
            push(pf[i:i + batch], False)
        n += len(pr)
    km, ks = k.compute()
    return dict(fid=float(f.compute()), kid=float(km), kid_std=float(ks), n_patches=n)


# ---------------------------------------------------------------- text legibility (OCR)
def edit_distance(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def ocr_reader():
    import easyocr
    import torch
    return easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)


def box_xyxy(bbox, H, W, pad=4):
    """easyocr quadrilateral -> padded, clipped axis-aligned box (x0, y0, x1, y1)."""
    xs = [p[0] for p in bbox]
    ys = [p[1] for p in bbox]
    return (max(0, int(min(xs)) - pad), max(0, int(min(ys)) - pad),
            min(W, int(max(xs)) + pad), min(H, int(max(ys)) + pad))


def detect_text(reader, img, min_conf=0.4, min_len=2):
    """[(box_xyxy, text)] for confident detections on the original image."""
    H, W = img.shape[:2]
    return [(box_xyxy(b, H, W), t) for b, t, c in reader.readtext(img) if c > min_conf and len(t.strip()) >= min_len]


def read_boxes(reader, img, boxes):
    return ["".join(t for _, t, _ in reader.readtext(img[y0:y1, x0:x1])) for x0, y0, x1, y1 in boxes]


def cer(refs, hyps) -> float:
    """Character error rate pooled over boxes; NaN if the reference has no characters."""
    n = sum(len(r) for r in refs)
    return sum(edit_distance(r, h) for r, h in zip(refs, hyps)) / n if n else float("nan")


def region_psnr(a, b, boxes) -> float:
    mask = np.zeros(a.shape[:2], bool)
    for x0, y0, x1, y1 in boxes:
        mask[y0:y1, x0:x1] = True
    return psnr_from_mse(float(((a[mask].astype(np.float64) - b[mask]) ** 2).mean())) if mask.any() else float("nan")


In [ ]:
%%writefile experiments/clic_b/b_analysis.py
"""Branch B: where does a DC-AE-based codec break on the CLIC validation set?

Steps (each writes into --out):
  inventory  per-image size, byte budgets, colour/flatness/edge stats, OCR text boxes,
             suggested label natural/screen          -> inventory.csv, budgets.json
  recon      reconstructions: DC-AE f32c32 ceiling, SD-VAE f8 ceiling,
             HEVC-intra proxy (x265, ~HM baseline) at 0.075/0.15/0.3 bpp -> recon/<cfg>/<name>.png
  metrics    PSNR, MS-SSIM, LPIPS, DISTS (tiled), text-region PSNR, OCR CER -> metrics.csv
  crops      256x256 crops around the main text box: orig | each cfg     -> crops/<name>.png
  summary    per-label means, CLIC-style dataset PSNR, B3 decision table   -> summary.md

python b_analysis.py all --data data/valid --out b_out
"""
from __future__ import annotations

import argparse
import json
import math
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw

sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "src"))  # repo checkout without `pip install -e .`
from ratflow.eval.metrics import (  # noqa: E402  (load/save/... re-exported for b6_vtm_residual.py)
    Perceptual, box_xyxy, cer, device, images, load, ocr_reader, read_boxes, read_csv, region_psnr, save, write_csv,
)

DATA_URL = "https://dhldkwazkze5h.cloudfront.net/data/clic2025_image_test.zip"
RATES = [0.075, 0.15, 0.3]
DCAE_ID = ("Efficient-Large-Model/Sana_1600M_1024px_diffusers", "vae")
SDVAE_IDS = [("stabilityai/sd-turbo", "vae"), ("stabilityai/sd-vae-ft-ema", None)]  # first = StableCodec's VAE
get_reader = ocr_reader


# ---------------------------------------------------------------- download
def download(data: Path):
    if data.exists() and images(data):
        return
    data.mkdir(parents=True, exist_ok=True)
    z = data / "valid.zip"
    subprocess.run(["wget", "-q", "-O", str(z), DATA_URL], check=True)
    with zipfile.ZipFile(z) as f:
        f.extractall(data)
    z.unlink()
    print("downloaded", len(images(data)), "images")


# ---------------------------------------------------------------- inventory
def inventory(args):
    download(args.data)
    reader = get_reader()
    rows, boxes = [], {}
    for p in images(args.data):
        x = load(p)
        H, W = x.shape[:2]
        rgb = x.reshape(-1, 3).astype(np.uint32)
        n_colors = len(np.unique((rgb[:, 0] << 16) | (rgb[:, 1] << 8) | rgb[:, 2]))
        g = x.astype(np.float32).mean(2)
        hb, wb = H // 8 * 8, W // 8 * 8
        blocks = g[:hb, :wb].reshape(hb // 8, 8, wb // 8, 8).std(axis=(1, 3))
        flat = float((blocks < 1.0).mean())
        grad = np.abs(np.diff(g, axis=1))[:-1] + np.abs(np.diff(g, axis=0))[:, :-1]
        sharp = float((grad > 96).mean())
        det = [(b, t, c) for b, t, c in reader.readtext(x) if c > 0.4 and len(t.strip()) >= 2]
        area = sum((x1 - x0) * (y1 - y0) for x0, y0, x1, y1 in (box_xyxy(b, H, W, 0) for b, _, _ in det))
        text_frac = area / (H * W)
        screenish = (n_colors / (H * W) < 0.05 and flat > 0.3) or text_frac > 0.03
        rows.append(dict(name=p.stem, H=H, W=W, pixels=H * W, n_colors=n_colors,
                         colors_per_px=round(n_colors / (H * W), 4), flat_frac=round(flat, 3),
                         sharp_edge_frac=round(sharp, 4), n_text_boxes=len(det),
                         text_area_frac=round(text_frac, 4),
                         label_suggested="screen" if screenish else "natural",
                         label="",  # <- fill in by hand: natural / game / screen
                         ))
        boxes[p.stem] = [[box_xyxy(b, H, W), t] for b, t, _ in det]
        print(rows[-1])
    total = sum(r["pixels"] for r in rows)
    budgets = {str(r): math.floor(r * total / 8) for r in RATES}
    args.out.mkdir(parents=True, exist_ok=True)
    write_csv(rows, args.out / "inventory.csv")
    json.dump(dict(total_pixels=total, budgets_bytes=budgets, boxes=boxes),
              open(args.out / "budgets.json", "w"), indent=1)
    print("total pixels", total, "budgets (bytes)", budgets)


# ---------------------------------------------------------------- reconstructions
def ae_roundtrip(args, cfg):
    import torch
    from diffusers import AutoencoderDC, AutoencoderKL

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if cfg == "dcae":
        vae = AutoencoderDC.from_pretrained(DCAE_ID[0], subfolder=DCAE_ID[1], torch_dtype=torch.float32)
        mult = 32
    else:
        vae = None
        for rid, sub in SDVAE_IDS:
            try:
                vae = AutoencoderKL.from_pretrained(rid, subfolder=sub, torch_dtype=torch.float32)
                print("SD-VAE:", rid)
                break
            except Exception as e:  # gated / missing -> next
                print("skip", rid, type(e).__name__)
        mult = 8
    vae = vae.to(dev).eval()
    vae.enable_tiling()
    for p in images(args.data):
        dst = args.out / "recon" / cfg / f"{p.stem}.png"
        if dst.exists():
            continue
        x = load(p)
        H, W = x.shape[:2]
        t = torch.from_numpy(x).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(dev)
        ph, pw = (-H) % mult, (-W) % mult
        t = torch.nn.functional.pad(t, (0, pw, 0, ph), mode="reflect")
        with torch.no_grad():
            if cfg == "dcae":
                y = vae.decode(vae.encode(t).latent).sample
            else:
                y = vae.decode(vae.encode(t).latent_dist.mode()).sample
        y = y[..., :H, :W].clamp(-1, 1).add(1).mul(127.5).round().byte()[0].permute(1, 2, 0).cpu().numpy()
        save(y, dst)
        print(cfg, p.stem)
    del vae
    torch.cuda.empty_cache()


def hevc_at(src: Path, target_bytes: int, dst: Path):
    """x265 intra, 4:4:4, fixed QP; smallest QP whose bitstream fits the budget.
    Proxy for the HM baseline (VTM is ~0.7 dB better at 0.075 bpp on the leaderboard)."""
    with tempfile.TemporaryDirectory() as td:
        bs = Path(td) / "o.hevc"

        def enc(qp):
            subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(src), "-frames:v", "1",
                            "-c:v", "libx265", "-pix_fmt", "yuv444p",
                            "-x265-params", f"qp={qp}:keyint=1:log-level=error", "-f", "hevc", str(bs)],
                           check=True)
            return bs.stat().st_size

        lo, hi, best = 0, 51, None
        while lo <= hi:
            qp = (lo + hi) // 2
            if enc(qp) <= target_bytes:
                best, hi = qp, qp - 1
            else:
                lo = qp + 1
        if best is None:
            best = 51
        size = enc(best)
        dst.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(bs), "-pix_fmt", "rgb24", str(dst)],
                       check=True)
    return best, size


def recon(args):
    for cfg in ["dcae", "sdvae"]:
        ae_roundtrip(args, cfg)
    if shutil.which("ffmpeg") is None or "libx265" not in subprocess.run(
            ["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout:
        print("WARNING: ffmpeg/libx265 missing -> no HEVC proxy")
        return
    log = []
    for p in images(args.data):
        x = load(p)
        H, W = x.shape[:2]
        for r in RATES:
            dst = args.out / "recon" / f"hevc@{r}" / f"{p.stem}.png"
            qp, size = hevc_at(p, math.floor(r * H * W / 8), dst)
            log.append(dict(name=p.stem, rate=r, qp=qp, bytes=size, bpp=round(size * 8 / (H * W), 4)))
            print(log[-1])
    write_csv(log, args.out / "hevc_log.csv")


# ---------------------------------------------------------------- metrics
def configs(out: Path):
    return sorted(d.name for d in (out / "recon").iterdir() if d.is_dir())


def metrics(args):
    perc = Perceptual(device())
    reader = get_reader()
    boxes = json.load(open(args.out / "budgets.json"))["boxes"]
    rows = []
    for p in images(args.data):
        x = load(p)
        bx = [b for b, _ in boxes.get(p.stem, [])]
        # OCR of the original, per box (same crop protocol as for the reconstructions)
        ref_txt = read_boxes(reader, x, bx)
        for cfg in configs(args.out):
            q = args.out / "recon" / cfg / f"{p.stem}.png"
            if not q.exists():
                continue
            y = load(q)
            row = dict(name=p.stem, cfg=cfg, **perc.all(x, y))
            if bx:
                row["text_psnr"] = region_psnr(x, y, bx)
                n_ch = sum(len(r) for r in ref_txt)
                if n_ch:
                    row["ocr_cer"] = cer(ref_txt, read_boxes(reader, y, bx))
                    row["ocr_chars"] = n_ch
            rows.append(row)
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
    write_csv(rows, args.out / "metrics.csv")


# ---------------------------------------------------------------- crops
def crops(args, size=256):
    boxes = json.load(open(args.out / "budgets.json"))["boxes"]
    cfgs = configs(args.out)
    for p in images(args.data):
        x = load(p)
        H, W = x.shape[:2]
        bx = boxes.get(p.stem, [])
        if bx:
            x0, y0, x1, y1 = max(bx, key=lambda b: (b[0][2] - b[0][0]) * (b[0][3] - b[0][1]))[0]
            cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
        else:
            cy, cx = H // 2, W // 2
        top = int(np.clip(cy - size // 2, 0, max(0, H - size)))
        left = int(np.clip(cx - size // 2, 0, max(0, W - size)))
        panels = [("orig", x)] + [(c, load(args.out / "recon" / c / f"{p.stem}.png")) for c in cfgs
                                  if (args.out / "recon" / c / f"{p.stem}.png").exists()]
        canvas = Image.new("RGB", (size * len(panels), size + 18), "white")
        d = ImageDraw.Draw(canvas)
        for i, (lab, a) in enumerate(panels):
            canvas.paste(Image.fromarray(a[top:top + size, left:left + size]), (i * size, 18))
            d.text((i * size + 4, 2), lab, fill="black")
        (args.out / "crops").mkdir(parents=True, exist_ok=True)
        canvas.save(args.out / "crops" / f"{p.stem}.png")


# ---------------------------------------------------------------- summary
def summary(args):
    inv = {r["name"]: r for r in read_csv(args.out / "inventory.csv")}
    rows = read_csv(args.out / "metrics.csv")
    lab = lambda n: inv[n]["label"] or inv[n]["label_suggested"]  # noqa: E731
    cfgs = sorted({r["cfg"] for r in rows})
    groups = ["all"] + sorted({lab(n) for n in inv})
    L = ["# Nhánh B: kết quả", ""]

    # CLIC-style dataset PSNR (MSE pooled, weighted by pixels)
    L += ["## PSNR toàn tập theo cách CLIC (MSE gộp, trọng số theo số pixel)", "",
          "Mốc so sánh trên leaderboard @0.075: HM 25.83, VTM 26.52, Vcoder (hạng 1) 23.99", "",
          "| cfg | PSNR |", "|---|---|"]
    for c in cfgs:
        rs = [r for r in rows if r["cfg"] == c]
        px = [int(inv[r["name"]]["pixels"]) for r in rs]
        mse = sum(float(r["mse"]) * n for r, n in zip(rs, px)) / sum(px)
        L.append(f"| {c} | {10 * math.log10(255 ** 2 / mse):.2f} |")

    def mean(rs, k):
        v = [float(r[k]) for r in rs if r.get(k) not in (None, "")]
        return (sum(v) / len(v), len(v)) if v else (float("nan"), 0)

    for g in groups:
        L += ["", f"## Nhóm: {g}", "", "| cfg | PSNR | MS-SSIM | LPIPS↓ | DISTS↓ | text PSNR | OCR CER↓ (n ảnh) |",
              "|---|---|---|---|---|---|---|"]
        for c in cfgs:
            rs = [r for r in rows if r["cfg"] == c and (g == "all" or lab(r["name"]) == g)]
            if not rs:
                continue
            cer, n = mean(rs, "ocr_cer")
            L.append(f"| {c} | {mean(rs, 'psnr')[0]:.2f} | {mean(rs, 'msssim')[0]:.4f} | {mean(rs, 'lpips')[0]:.4f} "
                     f"| {mean(rs, 'dists')[0]:.4f} | {mean(rs, 'text_psnr')[0]:.2f} | {cer:.3f} ({n}) |")

    # B3 decision helpers
    def m(cfg, k, grp):
        rs = [r for r in rows if r["cfg"] == cfg and lab(r["name"]) in grp]
        return mean(rs, k)[0]

    scr = {"screen", "game"}
    VTM_OFFSET = 1.9  # VTM (26.52) - our x265 proxy (24.58) at 0.075, dataset PSNR
    L += ["", "## Kiểm tra điều kiện B3 (HEVC dùng thay VTM, cộng thêm 1.9 dB)", ""]
    if "hevc@0.075" in cfgs:
        c1 = m("dcae", "ocr_cer", scr) <= m("hevc@0.075", "ocr_cer", scr)
        c2 = m("dcae", "psnr", scr) >= m("hevc@0.3", "psnr", scr) + VTM_OFFSET - 1
        c3 = m("sdvae", "ocr_cer", scr) <= m("hevc@0.075", "ocr_cer", scr)
        c4 = m("dcae", "psnr", {"natural"}) < m("hevc@0.3", "psnr", {"natural"}) + VTM_OFFSET - 2
        L += [f"- [screen/game] CER trần của DC-AE ≤ CER HEVC@0.075: **{c1}**",
              f"- [screen/game] PSNR trần của DC-AE ≥ VTM@0.3 − 1 dB: **{c2}**",
              f"- [screen/game] CER trần của f8 ≤ CER HEVC@0.075: **{c3}**",
              f"- [natural] PSNR trần của DC-AE < VTM@0.3 − 2 dB (cần residual ở rate cao): **{c4}**", ""]
        if c1 and c2:
            L.append("→ **Không cần nhánh riêng**; chỉ cần phân bổ nhiều bit hơn cho ảnh có chữ (A3).")
        elif c3:
            L.append("→ **Thêm nhánh residual / latent f8** cho ảnh có chữ và ảnh màn hình.")
        else:
            L.append("→ **Thêm mode SCC cổ điển** (VVC) cho ảnh màn hình.")
        if c4:
            L.append("→ **Ảnh tự nhiên ở 0.15–0.3 bpp cần đường residual** (cũng ảnh hưởng tới CVPR).")
    L += ["", "*Chú ý: `label` trong inventory.csv cần được xác nhận bằng mắt. Nhãn hiện tại có thể chỉ là nhãn gợi ý tự động.*"]
    (args.out / "summary.md").write_text("\n".join(L), encoding="utf-8")
    print("\n".join(L))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["inventory", "recon", "metrics", "crops", "summary", "all"])
    ap.add_argument("--data", type=Path, default=Path("data/valid"))
    ap.add_argument("--out", type=Path, default=Path("b_out"))
    args = ap.parse_args()
    steps = ["inventory", "recon", "metrics", "crops", "summary"] if args.step == "all" else [args.step]
    for s in steps:
        print(f"===== {s}")
        globals()[s](args)


if __name__ == "__main__":
    main()


In [ ]:
!pip install -q -U diffusers accelerate lpips piq easyocr
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -q libx265 && echo 'libx265 OK' || echo 'NO libx265 -> HEVC proxy skipped'

In [ ]:
B = 'experiments/clic_b/b_analysis.py'
!python {B} inventory

In [ ]:
!python {B} recon

In [ ]:
!python {B} metrics

In [ ]:
!python {B} crops

In [ ]:
!python {B} summary

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('b_out/summary.md', encoding='utf-8').read()))

In [ ]:
!cd b_out && zip -qr ../b_results.zip *.csv *.json *.md crops && ls -la ../b_results.zip